# The Sound of Data
## Analisi del dataset Spotify

Questo notebook verrà sviluppato passo passo. Prima di scegliere l'algoritmo, verificare sulle slide del corso quale metodo è richiesto come modello della percezione umana.


# 1. Obiettivo del progetto


A partire da un dataset di brani musicali estratti da Spotify, ciascuno corredato da caratteristiche sonore e musicali e associato a uno specifico genere, il progetto si propone di analizzare il modo in cui i brani possono essere organizzati e raggruppati sulla base delle loro proprietà audio.

In particolare, l'obiettivo è indagare se la suddivisione dei brani in generi musicali rispecchi una struttura riconoscibile, basata su caratteristiche musicali e acustiche misurabili, oppure se sia il risultato di un processo di categorizzazione cognitiva e percettiva operato dalla mente umana.

Attraverso l'applicazione di tecniche di clustering, si intende individuare gruppi di brani caratterizzati da proprietà sonore simili, considerando variabili come danceability, energy, valence, acousticness e tempo, senza utilizzare direttamente il genere musicale nella fase di raggruppamento.

I risultati ottenuti verranno successivamente analizzati e confrontati con i generi musicali originali, al fine di individuare eventuali corrispondenze o discrepanze tra le caratteristiche oggettive dei brani e la loro classificazione.

Il progetto mira quindi a comprendere come le caratteristiche sonore possano contribuire all'organizzazione della musica in categorie e quale relazione esista tra la struttura dei dati e i processi di percezione e categorizzazione umana.


In [ ]:
!pip install -q kagglehub

In [ ]:
# Librerie principali
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# 2. Caricamento del dataset



In questa fase carichiamo il dataset Spotify Tracks Dataset, disponibile su Kaggle.

Utilizzeremo la libreria `kagglehub` per scaricare automaticamente il dataset nell'ambiente Google Colab. Successivamente, leggeremo il file CSV tramite Pandas e controlleremo le dimensioni e la struttura iniziale dei dati.


In [ ]:
import kagglehub
from pathlib import Path

# Download automatico del dataset
dataset_path = kagglehub.dataset_download(
    "saichaitanyareddyai/spotify-tracks-dataset-audio-features"
)

# Ricerca del file CSV
csv_files = list(Path(dataset_path).rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nessun file CSV trovato nel dataset.")

DATA_PATH = csv_files[0]

# Caricamento del dataset
df = pd.read_csv(DATA_PATH)

print("Dataset caricato correttamente!")
print("File:", DATA_PATH.name)
print("Dimensioni:", df.shape)

display(df.head())

# Verifica delle colonne e dei tipi di dato
print("Colonne del dataset:")
print(df.columns.tolist())

print("\nTipi di dato:")
display(df.dtypes)


# 3. Panoramica e qualità dei dati



*TODO: colonne, tipi, valori mancanti, duplicati e distribuzione delle label.*

---

## 3.1 Dati identificativi

### `Track_id`

È il codice identificativo univoco assegnato da Spotify alla specifica traccia presente nel catalogo. È una stringa alfanumerica utilizzata per identificare il brano e recuperare le relative informazioni attraverso i sistemi Spotify. Non rappresenta una caratteristica musicale.

### `Artists`

Indica il nome dell'artista o degli artisti che interpretano il brano. Nel caso di collaborazioni, possono essere presenti più nomi associati alla stessa traccia.

### `Album_name`

Indica il nome dell'album all'interno del quale è pubblicato il brano. Può riferirsi a un album, un EP, un singolo o un'altra tipologia di pubblicazione.

### `Track_name`

Indica il titolo ufficiale del brano così come registrato nel catalogo Spotify.

---

## 3.2 Informazioni generali

### `Popularity`

Indica il livello di popolarità del brano, espresso attraverso un punteggio intero compreso tra 0 e 100. Viene determinato da un algoritmo proprietario di Spotify, basato principalmente sul numero complessivo di riproduzioni (stream) e sulla loro recenza. I brani ascoltati frequentemente nel periodo più recente tendono ad avere un punteggio maggiore rispetto a quelli che hanno ricevuto molti ascolti in passato. Il valore non viene necessariamente aggiornato in tempo reale e può presentare alcuni giorni di ritardo. Spotify non rende pubblica la formula matematica utilizzata.

### `Duration_ms`

Indica la durata complessiva del brano, espressa in millisecondi. È un dato numerico relativo alla lunghezza della registrazione presente nel catalogo Spotify.

### `Explicit`

Indica la presenza di contenuti espliciti nel brano, attraverso un valore booleano: `True` se il brano è contrassegnato come contenente contenuti espliciti, `False` in caso contrario. La classificazione è un'informazione associata alla traccia da Spotify e non una misura numerica ricavata dalle altre caratteristiche audio. È da considerare che, secondo la documentazione Spotify, False può indicare anche una condizione non conosciuta.

### `Track_genre`

Identifica il genere musicale associato al brano, attraverso un'etichetta testuale. È una variabile categorica che permette di raggruppare le tracce in base al genere di appartenenza. Nel dataset rappresenta un'etichetta associata al brano e non un punteggio ricavato direttamente dalle caratteristiche audio. Non è documentata una formula universale di Spotify per determinare questa classificazione.

---

## 3.3 Caratteristiche audio

### `Danceability`

Indica quanto un brano sia adatto al ballo, con un valore compreso tra 0 e 1. Viene stimata considerando una combinazione di elementi musicali, tra cui il tempo (velocità), la stabilità ritmica, la forza dei battiti (beat strength) e la regolarità complessiva del ritmo. Un valore vicino a 1 indica una maggiore predisposizione al ballo, mentre un valore vicino a 0 indica una minore predisposizione.

### `Energy`

Misura il livello di intensità ed energia percepita del brano, con un valore compreso tra 0 e 1. È una caratteristica percettiva che tiene conto di diversi elementi sonori, tra cui l'ampiezza della dinamica (dynamic range), il volume percepito, il timbro, la frequenza degli attacchi sonori e l'entropia generale del segnale. Brani caratterizzati da suoni intensi, veloci e incisivi tendono ad avere valori elevati, mentre composizioni più delicate e meno intense tendono ad avere valori inferiori.

### `Loudness`

Rappresenta il livello di intensità sonora complessiva del brano, espresso in decibel (dB). Spotify fornisce una misura mediata sull'intera durata della traccia, utile per confrontare il livello sonoro relativo tra diversi brani. I valori si collocano generalmente tra -60 e 0 dB, ma possono assumere anche valori positivi. Valori numericamente maggiori indicano un livello sonoro più elevato rispetto al riferimento della misurazione.

### `Speechiness`

Misura la presenza di contenuti vocali parlati all'interno del brano, con un valore compreso tra 0 e 1. Valori inferiori a 0,33 indicano generalmente contenuti prevalentemente musicali; valori compresi tra 0,33 e 0,66 possono indicare una combinazione bilaciata di musica e parlato, come nel rap; valori superiori a 0,66 identificano brani caratterizzati prevalentemente da contenuti parlati.

### `Acousticness`

Indica il grado di affidabilità con cui Spotify identifica un brano come acustico, attraverso un valore compreso tra 0 e 1. Non rappresenta necessariamente una probabilità statistica in senso stretto, ma un indice di confidenza: valori vicini a 1 indicano una maggiore sicurezza nell'identificazione del brano come acustico, mentre valori vicini a 0 indicano caratteristiche meno riconducibili a una registrazione acustica.

### `Instrumentalness`

Stima la probabilità che il brano non contenga parti vocali, attraverso un valore compreso tra 0 e 1. Valori vicini a 1 indicano una maggiore probabilità che si tratti di un brano strumentale. Spotify considera strumentali anche alcune vocalizzazioni prive di parole, mentre riconosce il rap e il parlato come contenuti vocali. Valori superiori a 0,5 sono generalmente indicativi di brani strumentali, con una maggiore affidabilità man mano che il valore si avvicina a 1.

### `Liveness`

Indica la probabilità che il brano sia stato registrato durante un'esibizione dal vivo, con un valore compreso tra 0 e 1. La stima si basa sul rilevamento della presenza di un pubblico nella registrazione. Valori più elevati indicano una maggiore probabilità di una registrazione dal vivo; valori superiori a 0,8 costituiscono un forte indicatore di questa condizione.

### `Valence`

Rappresenta la positività emotiva trasmessa musicalmente dal brano, attraverso un valore compreso tra 0 e 1. Valori elevati identificano sonorità generalmente più positive, allegre, euforiche o vivaci, mentre valori bassi indicano sonorità percepite come più negative, malinconiche, tristi o cupe. Si riferisce alla positività musicale percepita e non necessariamente al significato del testo.

### `Tempo`

Indica la velocità complessiva del brano, espressa in battiti per minuto (BPM, Beats Per Minute). Viene stimata attraverso l'analisi del ritmo e della durata media dei battiti. Valori elevati indicano generalmente un ritmo più veloce, mentre valori bassi indicano un ritmo più lento.

---

## 3.4 Caratteristiche musicali

### `Key`

Indica la tonalità musicale predominante del brano, rappresentata da un valore intero compreso tra 0 e 11, secondo il sistema di notazione Pitch Class. Ogni valore corrisponde a una delle dodici note della scala cromatica: `0 = Do (C), 1 = Do♯/Re♭, 2 = Re (D), 3 = Re♯/Mi♭, 4 = Mi (E), 5 = Fa (F), 6 = Fa♯/Sol♭, 7 = Sol (G), 8 = Sol♯/La♭, 9 = La (A), 10 = La♯/Si♭, 11 = Si (B)`. Spotify stima la tonalità attraverso l'analisi del contenuto sonoro e delle altezze delle note. Se non viene identificata una tonalità, la documentazione prevede anche il valore -1.

### `Mode`

Indica la modalità della tonalità musicale, distinguendo tra modo maggiore e modo minore. È rappresentata da un valore binario: 1 corrisponde al modo maggiore, mentre 0 corrisponde al modo minore. Questa informazione riguarda il tipo di scala musicale da cui deriva il contenuto melodico del brano.

### `Time_signature`

Indica la divisione metrica del brano, ovvero il numero di battiti presenti all'interno di ciascuna battuta musicale. È espressa attraverso un valore intero stimato da Spotify. Ad esempio, il valore 3 corrisponde a un tempo di 3/4, il valore 4 a un tempo di 4/4 e il valore 5 a un tempo di 5/4. La documentazione Spotify prevede valori compresi tra 3 e 7, che rappresentano il numeratore della divisione metrica.


# 4. Piano delle visualizzazioni


I grafici sono suddivisi in due categorie:

- **PRINCIPALE:** grafico che contribuisce direttamente al raggiungimento dell'obiettivo della ricerca.
- **FACOLTATIVO:** grafico utile per approfondire alcuni aspetti dell'analisi, ma non indispensabile.

## 4.1 Conoscere e analizzare il dataset

### 1. Numero di brani per genere — PRINCIPALE

- **Tipo di grafico:** Bar Chart orizzontale.
- **Dati utilizzati:** `track_genre` e numero di brani appartenenti a ciascun genere.

- **Obiettivo:**
  - Analizzare la distribuzione dei brani tra i diversi generi musicali.
  - Individuare i generi più e meno rappresentati.
  - Verificare la presenza di eventuali squilibri nel dataset.

- **Motivazione:**
  - Il grafico a barre orizzontali consente di confrontare facilmente le quantità associate alle diverse categorie.
  - L'orientamento orizzontale facilita la lettura dei nomi dei generi musicali.
  - L'ordinamento delle barre permette di individuare immediatamente i generi maggiormente rappresentati.
  - Questa visualizzazione fornisce una prima panoramica della composizione del dataset e aiuta a interpretare le analisi successive.

### 2. Distribuzione delle caratteristiche musicali — PRINCIPALE

- **Tipo di grafico:** Istogrammi, possibilmente organizzati in Small Multiples.
- **Dati utilizzati:**
  - `danceability`
  - `energy`
  - `valence`
  - `acousticness`
  - `instrumentalness`
  - `speechiness`
  - `tempo`
  - `loudness`

- **Obiettivo:**
  - Analizzare la distribuzione delle otto caratteristiche musicali.
  - Individuare gli intervalli di valori più frequenti.
  - Osservare eventuali distribuzioni asimmetriche e valori estremi.

- **Motivazione:**
  - L'istogramma è adatto a rappresentare la distribuzione di una variabile numerica e permette di osservare la frequenza dei diversi intervalli di valori.
  - Consente di individuare eventuali asimmetrie, concentrazioni di valori e distribuzioni particolari.
  - L'utilizzo degli Small Multiples permette di confrontare le otto caratteristiche attraverso grafici separati, mantenendo una visualizzazione ordinata e facilmente leggibile.
  - Questa analisi è importante per comprendere le variabili che verranno utilizzate successivamente nel clustering.

### 3. Correlazione tra le caratteristiche musicali — PRINCIPALE

- **Tipo di grafico:** Correlation Heatmap.
- **Dati utilizzati:** le otto caratteristiche musicali del grafico precedente.

- **Obiettivo:**
  - Individuare le caratteristiche che tendono a variare insieme.
  - Analizzare eventuali correlazioni positive o negative.
  - Verificare se alcune variabili descrivono aspetti musicali simili.

- **Motivazione:**
  - La Heatmap utilizza una scala di colori per rappresentare l'intensità e la direzione delle correlazioni tra le variabili.
  - Permette di confrontare rapidamente tutte le coppie di caratteristiche musicali.
  - È particolarmente adatta quando si devono analizzare più variabili contemporaneamente, perché rende le relazioni più immediate rispetto a una semplice tabella numerica.
  - Questa visualizzazione aiuta a comprendere le relazioni tra le caratteristiche che verranno utilizzate nel clustering.

### 4. Differenze delle caratteristiche musicali tra generi — FACOLTATIVO

- **Tipo di grafico:** Box Plot.
- **Dati utilizzati:**
  - `track_genre`
  - Un sottoinsieme delle otto caratteristiche musicali.

- **Obiettivo:**
  - Confrontare le caratteristiche musicali tra diversi generi.
  - Osservare mediane, dispersione e valori estremi.
  - Individuare eventuali somiglianze e differenze tra le categorie musicali.

- **Motivazione:**
  - Il Box Plot permette di rappresentare la distribuzione di una variabile numerica attraverso mediana, quartili e valori anomali.
  - Consente di confrontare contemporaneamente le distribuzioni di una caratteristica tra diversi generi.
  - È utile per osservare eventuali differenze e sovrapposizioni tra le categorie originali.
  - Questa visualizzazione costituisce un primo approfondimento sulle caratteristiche acustiche dei generi, che verranno successivamente confrontati con i cluster individuati.
---

## 4.2 Clustering: individuare gruppi senza utilizzare il genere

### 1. Elbow Method — PRINCIPALE

- **Tipo di grafico:** Line Chart.
- **Dati utilizzati:**
  - Le otto caratteristiche musicali standardizzate.
  - Diversi valori del numero di cluster `K`.
  - Inertia / WCSS calcolata da K-Means (somma dei quadrati delle distanze dei punti dai rispettivi centroidi).

- **Obiettivo:**
  - Analizzare diversi valori di `K`.
  - Osservare come varia l'inerzia all'aumentare del numero di cluster.
  - Individuare un possibile valore di `K` per la suddivisione dei brani.

- **Motivazione:**
  - Il Line Chart permette di rappresentare l'andamento di una misura al variare di un parametro.
  - Consente di osservare come diminuisce la WCSS aumentando il numero di cluster.
  - Permette di individuare il possibile punto di gomito, nel quale l'aggiunta di ulteriori cluster produce una riduzione dell'inerzia progressivamente meno significativa.
  - Il grafico fornisce un primo criterio per scegliere il numero di gruppi, evitando una scelta completamente arbitraria.

### 2. Silhouette Score — PRINCIPALE

- **Tipo di grafico:** Line Chart.
- **Dati utilizzati:**
  - Le otto caratteristiche musicali standardizzate.
  - Diversi valori del numero di cluster `K`.
  - Silhouette Score calcolato per ciascun valore di `K`.

- **Obiettivo:**
  - Valutare la qualità della suddivisione dei dati in cluster.
  - Analizzare quanto i brani appartenenti allo stesso gruppo siano simili tra loro.
  - Valutare quanto i cluster siano separati dagli altri gruppi.
  - Confrontare diverse configurazioni di `K`.

- **Motivazione:**
  - Il Line Chart permette di confrontare il Silhouette Score ottenuto con diversi valori di `K`.
  - La rappresentazione a linee rende immediatamente visibili gli aumenti e le diminuzioni del punteggio.
  - Il Silhouette Score considera sia la coesione interna dei gruppi sia la loro separazione dagli altri cluster.
  - Questa visualizzazione completa l'Elbow Method, offrendo un secondo criterio per valutare il numero di cluster più adeguato.

### 3. Visualizzazione dei cluster tramite PCA — PRINCIPALE

- **Tipo di grafico:** Scatter Plot.
- **Dati utilizzati:**
  - Le otto caratteristiche musicali standardizzate.
  - Le etichette dei cluster assegnate da K-Means.
  - Le prime due componenti principali ottenute tramite PCA.

- **Obiettivo:**
  - Rappresentare visivamente i gruppi individuati dall'algoritmo.
  - Osservare se i cluster appaiono distinti oppure sovrapposti.
  - Individuare eventuali gruppi vicini tra loro.

- **Motivazione:**
  - Lo Scatter Plot permette di rappresentare le singole osservazioni attraverso punti distribuiti in uno spazio bidimensionale.
  - Ogni punto rappresenta un brano musicale, mentre il colore identifica il cluster di appartenenza.
  - La PCA permette di ridurre le otto caratteristiche originali a due componenti principali, rendendo possibile la visualizzazione dei dati in due dimensioni.
  - Questa rappresentazione facilita l'osservazione della distribuzione dei brani e delle eventuali separazioni o sovrapposizioni tra i gruppi.

### 4. Profilo musicale dei cluster — PRINCIPALE

- **Tipo di grafico:** Heatmap.
- **Dati utilizzati:**
  - Etichette `cluster` ottenute da K-Means.
  - Le otto caratteristiche musicali.
  - Valore medio di ciascuna caratteristica per ogni cluster.

- **Obiettivo:**
  - Analizzare le caratteristiche musicali medie di ciascun cluster.
  - Individuare le variabili che contraddistinguono maggiormente ogni gruppo.
  - Attribuire un'interpretazione musicale ai cluster individuati.

- **Motivazione:**
  - La Heatmap permette di rappresentare contemporaneamente più caratteristiche per ciascun cluster attraverso una matrice di colori.
  - La variazione cromatica facilita il confronto tra i valori medi delle diverse caratteristiche musicali.
  - Consente di individuare rapidamente quali proprietà risultano maggiormente presenti o meno rappresentate in ciascun gruppo.
  - Questa visualizzazione permette di interpretare musicalmente i cluster, andando oltre la semplice rappresentazione dei punti nello Scatter Plot.

### 5. Confronto tra cluster e generi musicali — PRINCIPALE

- **Tipo di grafico:** Heatmap.
- **Dati utilizzati:**
  - `cluster` assegnato da K-Means.
  - `track_genre` originale.
  - Conteggi e percentuali per ogni combinazione cluster-genere.

- **Obiettivo:**
  - Confrontare i cluster individuati autonomamente con i generi musicali originali.
  - Osservare quali generi sono presenti in ciascun cluster.
  - Individuare generi distribuiti in più cluster.
  - Individuare cluster che contengono brani appartenenti a generi differenti.
  - Analizzare quanto i gruppi acustici corrispondono alle categorie musicali originali.

- **Motivazione:**
  - La Heatmap permette di rappresentare la relazione tra due variabili categoriali: cluster e genere musicale.
  - L'utilizzo dei colori evidenzia quali combinazioni cluster-genere sono più frequenti.
  - Le percentuali per riga consentono di osservare la composizione interna di ciascun cluster, facilitando il confronto tra gruppi di dimensioni differenti.
  - Questa visualizzazione è fondamentale per rispondere alla domanda di ricerca, perché permette di verificare se i brani raggruppati in base alle caratteristiche acustiche corrispondono ai generi originali oppure se generi diversi vengono riuniti negli stessi cluster.

---

## 4.3 Classificazione supervisionata: prevedere il genere musicale — FACOLTATIVA

Questa fase costituisce un approfondimento rispetto all'obiettivo principale. A differenza del clustering, che individua gruppi senza conoscere in anticipo i generi, la classificazione supervisionata utilizza le etichette originali come variabile da prevedere.

### 1. Decision Tree — FACOLTATIVO

- **Tipo di visualizzazione:** Albero decisionale (Decision Tree).
- **Dati utilizzati:**
  - Input: le otto caratteristiche musicali.
  - Target: `track_genre`.

- **Obiettivo:**
  - Costruire un modello in grado di prevedere il genere musicale di un brano.
  - Osservare quali caratteristiche vengono utilizzate durante le decisioni.
  - Comprendere le regole utilizzate dal modello per classificare le tracce.

- **Motivazione:**
  - L'albero decisionale rappresenta le regole di classificazione attraverso nodi e rami.
  - Permette di seguire il percorso delle decisioni e comprendere quali caratteristiche vengono utilizzate per assegnare un genere a un brano.
  - È una visualizzazione interpretabile, utile per comprendere il funzionamento del modello e confrontare i risultati con quelli ottenuti attraverso il clustering.

### 2. Feature Importance — FACOLTATIVO

- **Tipo di grafico:** Bar Chart orizzontale.
- **Dati utilizzati:**
  - Modello Random Forest.
  - Le otto caratteristiche musicali.
  - Valori di Feature Importance.

- **Obiettivo:**
  - Individuare le caratteristiche musicali più informative per la classificazione.
  - Ordinare le variabili in base alla loro importanza.
  - Comprendere quali proprietà musicali contribuiscono maggiormente alle previsioni del modello.

- **Motivazione:**
  - Il grafico a barre permette di confrontare e ordinare facilmente l'importanza delle diverse caratteristiche.
  - L'orientamento orizzontale facilita la lettura dei nomi delle variabili.
  - Consente di individuare rapidamente quali caratteristiche risultano più utili al modello per distinguere i generi musicali.

### 3. Confronto Decision Tree vs Random Forest — FACOLTATIVO

- **Tipo di grafico:** Bar Chart comparativo.
- **Dati utilizzati:**
  - Risultati del Decision Tree.
  - Risultati della Random Forest.
  - Metriche di valutazione calcolate sullo stesso test set.

- **Obiettivo:**
  - Confrontare le prestazioni dei due algoritmi.
  - Individuare quale modello ottiene risultati migliori nella previsione dei generi.
  - Valutare se un modello più complesso offre un miglioramento rispetto a un singolo albero decisionale.

- **Motivazione:**
  - Il grafico a barre permette di confrontare direttamente le prestazioni dei due modelli.
  - Utilizzando la stessa scala è possibile osservare facilmente le differenze tra i risultati.
  - Offre una rappresentazione immediata delle prestazioni dei modelli di classificazione.

### 4. Confusion Matrix — FACOLTATIVO

- **Tipo di grafico:** Confusion Matrix / Heatmap.
- **Dati utilizzati:**
  - Generi reali presenti nel test set.
  - Generi previsti dal modello.
  - Risultati del Decision Tree o della Random Forest.

- **Obiettivo:**
  - Osservare quali generi vengono classificati correttamente.
  - Individuare quali generi vengono confusi tra loro.
  - Analizzare eventuali difficoltà nella distinzione tra categorie musicali.

- **Motivazione:**
  - La Heatmap permette di rappresentare il confronto tra le classi reali e quelle previste dal modello.
  - L'utilizzo dei colori rende immediatamente visibili le classificazioni corrette e gli errori.
  - Consente di individuare quali generi vengono confusi più frequentemente e di approfondire i limiti del modello.


# 5. Preprocessing e normalizzazione


## 5.1 Selezione delle caratteristiche


## 5.2 Pulizia e preparazione dei dati



## 5.3 Standardizzazione delle variabili



## 5.4 Dataset finale per il clustering


# 6. Applicazione dell'algoritmo di clustering


6.1 Introduzione e scelta dell'algoritmo



6.2 Scelta del numero di cluster
- 6.2.1 Elbow Method
- 6.2.2 Silhouette Score



6.3 Applicazione di K-Means



6.4 Valutazione del clustering


# 7. Interpretazione e visualizzazione dei risultati


7.1 Visualizzazione tramite PCA



7.2 Profilo musicale dei cluster



7.3 Confronto tra cluster e generi musicali



7.4 Interpretazione dei risultati



7.5 Eventuali approfondimenti (facoltativi)


# 8. Conclusioni e limiti


8.1 Risultati principali



8.2 Risposta alla domanda di ricerca



8.3 Limiti dell'analisi



8.4 Possibili sviluppi futuri
